# Synthetic player movement heatmap

This notebook turns the layout of the `iteration_3` Mermaid diagram into a fictional game map. The central node becomes `spawn`, and the surrounding nodes become bounded areas connected by the same four-branch structure.

**All players, timestamps, coordinates, areas, and paths are synthetic.**

In [1]:
from pathlib import Path

import pandas as pd

from synthetic_heatmap import (
    area_frame,
    build_heatmap_figure,
    build_movement_animation,
    generate_player_paths,
    save_player_paths,
    validate_player_paths,
)

## 1. Generate a small reproducible dataset

Nine players are sampled every five seconds for ten minutes. Each path starts at `spawn`. A graph-based semi-Markov walk gives each player seeded route preferences and variable dwell times. Players then walk continuously along a valid connector at an individual bounded speed. This is more appropriate than MCMC because the goal is believable movement, not sampling a target posterior distribution.

In [2]:
player_paths = generate_player_paths(
    player_count=9,
    duration_minutes=10,
    sample_interval_seconds=5,
    seed=20261002,
)

print(f"Rows: {len(player_paths):,}")
print(f"Players: {player_paths['player_id'].nunique()}")
player_paths.head()

Rows: 1,089
Players: 9


,player_id,timestamp,elapsed_minutes,x,y,bbox_x_min,bbox_x_max,bbox_y_min,bbox_y_max,name,movement_state,target_name
0,1,2026-10-02 12:00:00+00:00,0.000000,50.000,30.000,46.0,54.0,26.0,34.0,spawn,dwelling,spawn
1,1,2026-10-02 12:00:05+00:00,0.083333,49.917,29.874,46.0,54.0,26.0,34.0,spawn,dwelling,spawn
2,1,2026-10-02 12:00:10+00:00,0.166667,49.998,29.674,46.0,54.0,26.0,34.0,spawn,dwelling,spawn
3,1,2026-10-02 12:00:15+00:00,0.250000,49.802,29.493,46.0,54.0,26.0,34.0,spawn,dwelling,spawn
4,1,2026-10-02 12:00:20+00:00,0.333333,49.741,29.531,46.0,54.0,26.0,34.0,spawn,dwelling,spawn


## 2. Validate the synthetic-data contract

The checks below prove that every path starts at `spawn`, every sampled coordinate is inside the bounding box repeated on that row, and consecutive samples never contain an unrealistic movement jump.

In [3]:
validate_player_paths(player_paths)

starts = (
    player_paths.sort_values(['player_id', 'timestamp'])
    .groupby('player_id', as_index=False)
    .first()
)

checks = pd.Series(
    {
        'all_players_start_at_spawn': starts['name'].eq('spawn').all(),
        'all_x_inside_bounds': player_paths['x'].between(
            player_paths['bbox_x_min'], player_paths['bbox_x_max']
        ).all(),
        'all_y_inside_bounds': player_paths['y'].between(
            player_paths['bbox_y_min'], player_paths['bbox_y_max']
        ).all(),
        'walking_samples_exist': player_paths['movement_state'].eq('walking').any(),
    },
    name='passed',
)
checks

all_players_start_at_spawn    True
all_x_inside_bounds           True
all_y_inside_bounds           True
walking_samples_exist         True
Name: passed, dtype: bool

## 3. Inspect the map and sampled activity

The area table is the fictional map definition. The summary counts position samples, not distinct players.

In [4]:
display(area_frame())

area_summary = (
    player_paths.groupby('name', as_index=False)
    .agg(position_samples=('player_id', 'size'), players=('player_id', 'nunique'))
    .sort_values('position_samples', ascending=False)
)
area_summary

,name,x_min,x_max,y_min,y_max,shape
0,spawn,46,54,26,34,circle
1,area1,34,46,39,46,rect
2,area2,24,34,50,56,rect
3,area3,44,56,50,56,rect
4,area4,14,25,38,44,rect
5,area5,64,79,33,40,rect
6,area6,66,76,47,53,rect
7,area7,82,92,43,49,rect
8,area8,88,99,28,35,rect
9,area9,28,41,27,34,rect


,name,position_samples,players
11,spawn,268,9
0,area1,261,5
3,area12,163,3
9,area3,88,4
8,area2,86,4
5,area14,45,2
7,area16,40,1
1,area10,40,1
10,area9,31,1
6,area15,28,1


## 4. Save the fake CSV

In [5]:
output_path = save_player_paths(
    player_paths,
    Path('data') / 'fake_player_paths.csv',
)
output_path

WindowsPath('data/fake_player_paths.csv')

## 5. Animate movement through elapsed time

The bounding boxes and connector lines retain the structure of the Mermaid visual. Use Play or drag the elapsed-time slider. Colourblind-safe markers show current positions, while short matching trails make direction and recent movement visible.

In [6]:
figure = build_movement_animation(
    player_paths,
    minute_range=(0.0, 10.0),
    trail_minutes=1.0,
    frame_interval_seconds=10,
    show_bounding_boxes=True,
)
figure.show()

## Interpretation

Darker cells represent more sampled positions. They do **not** represent unique players, deaths, difficulty, or performance. A real analysis would first choose the business question and then select the event grain and metric that answer it.